## **Customer Churn and Survival Analysis.**

## **Load data and drop leakage prone columns.**

In [1]:
import os
import pandas as pd
import numpy as np

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/CustomerChurn/"

data_df = pd.read_excel(DATASETS_PATH + "Telco_customer_churn.xlsx")

data_df = data_df.drop(columns=["Churn Score", "CLTV", "Churn Reason"])

print("Data: ")
print(data_df)


Data: 
      CustomerID  Count        Country       State          City  Zip Code  \
0     3668-QPYBK      1  United States  California   Los Angeles     90003   
1     9237-HQITU      1  United States  California   Los Angeles     90005   
2     9305-CDSKC      1  United States  California   Los Angeles     90006   
3     7892-POOKP      1  United States  California   Los Angeles     90010   
4     0280-XJGEX      1  United States  California   Los Angeles     90015   
...          ...    ...            ...         ...           ...       ...   
7038  2569-WGERO      1  United States  California       Landers     92285   
7039  6840-RESVB      1  United States  California      Adelanto     92301   
7040  2234-XADUH      1  United States  California         Amboy     92304   
7041  4801-JZAZL      1  United States  California  Angelus Oaks     92305   
7042  3186-AJIEK      1  United States  California  Apple Valley     92308   

                    Lat Long   Latitude   Longitude  Gen

## **Convert columns with string values that have more than one value to one-hot-encoding.**

In [2]:
data_df["Total Charges"] = pd.to_numeric(data_df["Total Charges"], errors="coerce")
data_df["Total Charges"] = data_df["Total Charges"].fillna(0.0)

data_df["Short_Term_Contract"] = (
    (data_df["Contract"] == "Month-to-month").astype(int)
)

data_df["Contract_Tenure_Risk"] = (
    data_df["Short_Term_Contract"] / (data_df["Tenure Months"] + 1)
)

risk_services = [
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support"
]

data_df["Missing_Services"] = (data_df[risk_services] == "No").sum(axis=1)

encoded_df = data_df.drop(columns=["Churn Value", "Churn Label", "CustomerID","Country","State","City","Lat Long", "Latitude", "Longitude", "Count", "Zip Code"])
y = data_df["Churn Value"]

for col in encoded_df.columns:
    if pd.api.types.is_string_dtype(data_df[col]):
        print(col)

Gender
Senior Citizen
Partner
Dependents
Phone Service
Multiple Lines
Internet Service
Online Security
Online Backup
Device Protection
Tech Support
Streaming TV
Streaming Movies
Contract
Paperless Billing
Payment Method


In [3]:
X_encoded = pd.get_dummies(encoded_df, drop_first=True)
print(X_encoded)

      Tenure Months  Monthly Charges  Total Charges  Short_Term_Contract  \
0                 2            53.85         108.15                    1   
1                 2            70.70         151.65                    1   
2                 8            99.65         820.50                    1   
3                28           104.80        3046.05                    1   
4                49           103.70        5036.30                    1   
...             ...              ...            ...                  ...   
7038             72            21.15        1419.40                    0   
7039             24            84.80        1990.50                    0   
7040             72           103.20        7362.90                    0   
7041             11            29.60         346.45                    1   
7042             66           105.65        6844.50                    0   

      Contract_Tenure_Risk  Missing_Services  Gender_Male  Senior Citizen_Yes  \
0     

In [4]:
print(X_encoded.columns)

Index(['Tenure Months', 'Monthly Charges', 'Total Charges',
       'Short_Term_Contract', 'Contract_Tenure_Risk', 'Missing_Services',
       'Gender_Male', 'Senior Citizen_Yes', 'Partner_Yes', 'Dependents_Yes',
       'Phone Service_Yes', 'Multiple Lines_No phone service',
       'Multiple Lines_Yes', 'Internet Service_Fiber optic',
       'Internet Service_No', 'Online Security_No internet service',
       'Online Security_Yes', 'Online Backup_No internet service',
       'Online Backup_Yes', 'Device Protection_No internet service',
       'Device Protection_Yes', 'Tech Support_No internet service',
       'Tech Support_Yes', 'Streaming TV_No internet service',
       'Streaming TV_Yes', 'Streaming Movies_No internet service',
       'Streaming Movies_Yes', 'Contract_One year', 'Contract_Two year',
       'Paperless Billing_Yes', 'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check'],
      dtype='str')


In [5]:
baseline_dict = {}

for col in encoded_df.select_dtypes(include=["string", "category"]).columns:
    categories = sorted(encoded_df[col].dropna().unique())
    baseline_dict[col] = categories[0]  # first one is dropped

print(baseline_dict)

{'Gender': 'Female', 'Senior Citizen': 'No', 'Partner': 'No', 'Dependents': 'No', 'Phone Service': 'No', 'Multiple Lines': 'No', 'Internet Service': 'DSL', 'Online Security': 'No', 'Online Backup': 'No', 'Device Protection': 'No', 'Tech Support': 'No', 'Streaming TV': 'No', 'Streaming Movies': 'No', 'Contract': 'Month-to-month', 'Paperless Billing': 'No', 'Payment Method': 'Bank transfer (automatic)'}


## **Divide into train and test datasets.**

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train["Charge_per_Tenure"] = (
    X_train["Monthly Charges"] / (X_train["Tenure Months"] + 1)
)

X_test["Charge_per_Tenure"] = (
    X_test["Monthly Charges"] / (X_test["Tenure Months"] + 1)
)



## **Conduct logistic regression for baseline.**

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.35

y_pred = (y_proba >= threshold).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

from sklearn.metrics import roc_auc_score

print("ROC-AUC: ")
roc_auc_score(y_test, y_proba)

              precision    recall  f1-score   support

           0       0.88      0.82      0.85      1009
           1       0.61      0.71      0.65       400

    accuracy                           0.79      1409
   macro avg       0.74      0.76      0.75      1409
weighted avg       0.80      0.79      0.79      1409

ROC-AUC: 


0.8599145193260654

## **Conduct classification with Random Forest.**

In [42]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=4,
    max_features="sqrt",
    bootstrap=True,
    max_samples=0.65,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))

print("ROC-AUC: ")
roc_auc_score(y_test, y_proba_rf)


              precision    recall  f1-score   support

           0       0.89      0.81      0.85      1009
           1       0.61      0.73      0.67       400

    accuracy                           0.79      1409
   macro avg       0.75      0.77      0.76      1409
weighted avg       0.81      0.79      0.80      1409

ROC-AUC: 


0.8567133300297324

## **Conduct classification with XGBoost model.**

In [21]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=15,
    subsample=0.7,
    colsample_bytree=0.8,
    reg_lambda=1,
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))

print("ROC-AUC:")
roc_auc_score(y_test, y_proba_xgb)

              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1009
           1       0.61      0.65      0.63       400

    accuracy                           0.78      1409
   macro avg       0.73      0.74      0.74      1409
weighted avg       0.79      0.78      0.78      1409

ROC-AUC:


0.8446469276511397

## **Conduct classification with lightGBM model.**

In [23]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=-1,          # -1 means no limit
    num_leaves=50,
    min_child_samples=20,
    subsample=0.7,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lgb))

print("ROC-AUC:")
roc_auc_score(y_test, y_proba_lgb)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1469, number of negative: 4165
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000177 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 971
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
              precision    recall  f1-score   support

           0       0.87      0.81      0.84      1009
           1       0.59      0.70      0.64       400

    accuracy                           0.78      1409
   macro avg       0.73      0.75      0.74      1409
weighted avg       0.79      0.78      0.78      1409

ROC-AUC:


0.8485517839444996